# Power budget — how it works

This is the explanation and reference for `quicksat`'s power budget. It covers how the generation and the loads meet through the power conditioning and the battery, where the losses sit, the checks, and where the model stops.

It is not a tutorial. The worked example is `sample/power/power_budget.ipynb`. This notebook still runs, so that it cannot drift from the code. It reads `docs/power/data/`, its own copy of the inputs. The figures quoted below are true of that data.

The generation and the consumption have references of their own: `docs/power/power_generation_ref.ipynb` and `docs/power/power_consumption_ref.ipynb`. An unsupported scenario is a power result, not a scenario error (`docs/explanations/decisions.md`, entry 10).

In [1]:
import os
from pathlib import Path

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import Q_
from quicksat.power.budget import LOSSES, budget_summary, run_budget
from quicksat.power.config import PowerSystem
from quicksat.power.consumption import PowerConsumption, run_consumption
from quicksat.power.generation import run_generation
from quicksat.scenario.config import Scenario
from quicksat.scenario.run import run_scenario

DATA = Path("docs") / "power" / "data"
run = run_scenario(Scenario.from_yaml_file(DATA / "scenario.yaml"))
power_system = PowerSystem.from_yaml_file(DATA / "power_system.yaml")
generation = run_generation(power_system, run)
loads = run_consumption(PowerConsumption.from_csv(DATA / "power_consumption.csv"), run)
budget = run_budget(power_system, generation, loads)


def with_changes(**changes) -> PowerSystem:
    """This notebook's power system with some keys changed."""
    return power_system.model_copy(update=changes)

## What the budget needs

Three files: the scenario, the power system file and the consumption file. The power system file's panels give the generation. Three more keys give the budget:

| Key | What it holds |
| --- | --- |
| `battery` | `capacity`, what it holds when new and full, such as `1024 W h`; `capacity_factor`, the share left in the year being sized; `max_depth_of_discharge`, the deepest it may go; and `initial_charge`, the charge at the start of the run, full by default |
| `conditioning` | The efficiency of each path: `array_regulator`, `battery_charge`, `battery_discharge` and `distribution` |
| `system_margin` | The margin on all the loads together, on top of each equipment's own |

astropy does not read `Wh`, so watt-hours are written `W h`. The budget stops if any of the three keys is missing. This notebook's values:

In [2]:
battery, conditioning = power_system.battery, power_system.conditioning
assert battery is not None and conditioning is not None
print(f"capacity       {battery.capacity:.0f}, {battery.capacity_factor:.2f} of it left: "
      f"{battery.usable:.1f} usable")
print(f"max depth      {battery.max_depth_of_discharge:.0f}")
print(f"conditioning   regulator {conditioning.array_regulator}, charge "
      f"{conditioning.battery_charge}, discharge {conditioning.battery_discharge}, "
      f"distribution {conditioning.distribution}")
print(f"system margin  {power_system.system_margin:.0f}")

capacity       1024 W h, 0.80 of it left: 819.2 W h usable
max depth      30 %
conditioning   regulator 0.96, charge 0.95, discharge 0.95, distribution 0.97
system margin  5 %


## The losses, in three places

Each loss is counted once, where it happens:

1. **Up to the array's output:** the power system file's `loss_factor`, in the generation. It takes the cells' temperature, ageing, mismatch between cells, UV and the array's own wiring.
2. **The power conditioning:** an efficiency for each path the energy takes after that. From the array to the bus through the array regulator. From the bus into the battery through the charge regulator and the cells. Back out through the discharge regulator and the cells. From the bus to the loads through the switches, the fuses and the harness.
3. **Fixed overheads, as loads:** the power unit's own electronics and the heaters are rows of the consumption file.

The budget follows the energy step by step over the run's grid. In each step:

- the loads, with their equipment margins and the system margin, need their energy over `distribution` at the bus;
- the array's energy reaches the bus times `array_regulator`;
- a surplus charges the battery, times `battery_charge`, until it is full. What a full battery cannot take is unused: the array is held off its best point then;
- a shortfall comes from the battery, which gives it at `battery_discharge`. If the battery is empty, the rest is unmet.

So energy that passes through the battery pays both its paths: about 10% with this notebook's efficiencies, on top of the regulator and the distribution. A single efficiency for the whole system would hide that, and would understate a long eclipse. For a bus fed straight from the battery, with no charge and discharge regulators, give the cells' own efficiencies, about 0.97 each.

The energy over a step is exact for the generation, which changes linearly between the steps, and for the loads, which change at exact times between them. Every joule is accounted for. Over the run, in W h:

In [3]:
flows = {name: flow.sum() / 3600 for name, flow in budget.flows.items()}
print(f"generated at the array's output  {flows['generated']:8.1f}")
print(f"drawn from the battery           {flows['drawn']:8.1f}")
print(f"consumed by the loads            {flows['consumed']:8.1f}")
for name in LOSSES:
    print(f"lost in {name.replace('_', ' '):25s}{flows[name]:8.1f}")
print(f"stored in the battery            {flows['stored']:8.1f}")
print(f"unused                           {flows['unused']:8.1f}")
come = flows["generated"] + flows["drawn"]
go = flows["consumed"] + flows["stored"] + flows["unused"] + sum(flows[n] for n in LOSSES)
print(f"in less out                      {come - go:8.1e}")

generated at the array's output    1302.1
drawn from the battery              432.3
consumed by the loads              1116.0
lost in array regulator              48.6
lost in battery charge               21.3
lost in battery discharge            21.6
lost in distribution                 34.5
stored in the battery               404.8
unused                               87.6
in less out                      -2.3e-13


## The battery

The battery holds its capacity times its capacity factor when full, in the year being sized. Its charge is a share of that. The budget starts it at `initial_charge`.

**Sizing from the mass.** This notebook's battery is the mass budget's 12.5 kg of Saft VES16 cells. A VES16 cell holds 4.5 A h and about 16 W h. Saft's 8s4p pack, 32 cells, is listed at 5.8 kg and 512 W h, about 88 W h/kg. These figures come from distributors, not from Saft's datasheet. 12.5 kg is then about 64 cells: 8 in series for a 28 V bus, and 8 strings in parallel, 1024 W h. At the end of life 80% is left, 819 W h. A 30% depth of discharge is a usual limit in low orbit, where the battery cycles some 5500 times a year. An 8s4p pack alone would go about 33% deep in each eclipse at the end of life, past the limit, with nothing left for a safe mode.

The charge over the run, with the deepest point of each orbit:

In [4]:
budget.orbit_table[["orbit", "charge at start [%]", "lowest charge [%]", "charge at end [%]"]].style.hide(
    axis="index"
).format("{:.1f}", subset=["charge at start [%]", "lowest charge [%]", "charge at end [%]"])

orbit,charge at start [%],lowest charge [%],charge at end [%]
1,100.0,83.6,96.6
2,96.6,83.6,96.6
3,96.6,83.6,96.6


## The checks

- **Battery charge:** the charge never falls below 1 less the maximum depth of discharge.
- **Energy balance:** each whole orbit has a margin of zero or more. The margin is the net energy into the battery, with the unused energy counted as if the battery had taken it: times `array_regulator` and `battery_charge`. So it does not depend on where in the orbit the window starts, nor on how full the battery is.
- **Loads met:** the battery never runs flat.

The first check alone is not enough. A run that starts with the battery full can stay above the limit for a few orbits while it loses energy every orbit. With a system margin of 10% instead of 5%, the charge stays above 70% here, but the margin turns negative:

In [5]:
for margin in ("5 %", "10 %", "15 %"):
    other = run_budget(with_changes(system_margin=Q_(margin)), generation, loads)
    statuses = ", ".join(f"{check.name} {check.status}" for check in other.checks)
    margins = other.orbit_table["margin [Wh]"].round(1).tolist()
    print(f"{margin:>5}: lowest {100 * other.charge.min():.1f}%, margins {margins} W h: {statuses}")

  5 %: lowest 83.6%, margins [17.6, 18.3, 16.5] W h: Battery charge ok, Energy balance ok, Loads met ok
 10 %: lowest 82.6%, margins [-0.6, 0.0, -1.6] W h: Battery charge ok, Energy balance fail, Loads met ok
 15 %: lowest 77.4%, margins [-18.9, -18.2, -19.8] W h: Battery charge ok, Energy balance fail, Loads met ok


The run's `checks` gives each check's status and a sentence, which the viewer shows. `budget_summary` gives the figures the viewer lists:

In [6]:
for item in budget_summary(budget):
    print(f"{item['label']:32s}{item['value']:>8} {item['unit']}")

Lowest charge                       83.6 %
Mean generated                       275 W
Mean consumed                        235 W
Usable capacity                      819 W h
Deepest discharge                   16.4 %
Unused per orbit                      29 W h
Conditioning losses per orbit         42 W h
Peak load                            340 W


## The viewer

`write_power_viewer` writes a folder with the scenario viewer's page as `power_viewer.html`, the run's `scenario.js`, and `power.js`. The page shows the power only when `power.js` carries the `id` of the `scenario.js` beside it, so both come from one run. `power.js` holds:

- the power system file's name and text;
- the body box and each panel, as the model draws it, for a page without a 3D model;
- on the run's grid: the power generated and consumed, the charge, and each panel's power and its cosine to the sun, scaled by the light;
- the battery's depth limit, which the page draws as a line, and its capacities;
- the setup panel's blocks, the summary and the checks.

Write each scenario's budget to a folder of its own, such as one for winter and one for summer.

## Limitations

What the budget does not do, yet or at all:

- **Energy, not current.** The battery is an energy store. Its voltage, its current, the charge rate and its temperature are not modelled, so a charge limit such as C/2 is not checked.
- **Steady efficiencies.** Each path keeps one efficiency, whatever its load.
- **Unused energy is lost.** Once the battery is full, the array's spare power is not used, as a shunt or an off-peak array regulator would do.
- **No safe mode by itself.** A safe mode needs its own activities and column, like any mode.
- **Short runs.** A run covers an orbit to a day or two. Seasons and the worst beta angle need a scenario each, such as winter and summer, each with its own budget.